[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Confusion_Matrix_Pair.ipynb)

# Beyond Accuracy: Which Mistakes Is the Model Making?
**CS 377 · companion to 18_1_1 Classification Foundations and 18_1_2 The Confusion Matrix · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Beyond Accuracy quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1; loading the data and training the model takes a few seconds.

It rebuilds the German Credit model from 18_1_1 and 18_1_2 (the same split and the same XGBoost settings) and makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 700 training and 300 test applicants; in `y`, 1 means the applicant defaulted and 0 means they repaid. 90 of the 300 test applicants defaulted.
- `SPW`: the `scale_pos_weight` from 18_1_1, 490 / 210 ≈ 2.33.
- `make_model(weight)`: the 18_1_1 XGBoost model with `scale_pos_weight=weight`, not yet trained.
- `model`, `y_pred`, `y_proba`: the trained weight-2.33 model (the 18_1_1 model), its 0/1 predictions at the 0.5 threshold, and its predicted probability of default for each test applicant.

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # hides library warnings, including the one Q8's precision would trigger

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

pd.set_option('display.width', 140)   # keep the Q5 table on one line

# German Credit, prepared exactly as in 18_1_1 and 18_1_2
df = fetch_openml(name='credit-g', version=1, as_frame=True).frame
y = (df['class'] == 'bad').astype(int)        # 1 = defaulted (bad credit), 0 = repaid (good credit)
X = df.drop(columns=['class'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

SPW = (y_train == 0).sum() / (y_train == 1).sum()   # scale_pos_weight: 490 / 210, about 2.33


def make_model(weight):
    """The 18_1_1 XGBoost model, with scale_pos_weight set to `weight`."""
    return xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, scale_pos_weight=weight,
                             eval_metric='logloss', random_state=42, enable_categorical=True, tree_method='hist')


model = make_model(SPW).fit(X_train, y_train)
y_pred = model.predict(X_test)                 # 0/1 predictions at the 0.5 threshold
y_proba = model.predict_proba(X_test)[:, 1]    # predicted probability of default

print(f"training applicants: {len(y_train)}   test applicants: {len(y_test)}   "
      f"test defaulters: {y_test.sum()}   scale_pos_weight: {SPW:.2f}")

# Part 2 (Q3-Q6): The German Credit Model

## Q3 · Count the four cells
> **Quiz Q3:** Fill in `FP`, `FN` and `TN` the same way the cell computes `TP`, and run it. What are the four counts? Check them against the confusion matrix in 18_1_2. scikit-learn prints them in the order `tn, fp, fn, tp`: why that order? (Look at the matrix in Q2.) Which cell holds the defaulters the bank approved?

**✏️ Your change:** Replace each `np.nan` with a mask like the one for `TP`, changing the 1s and 0s on the right of each `==`.

**Tell your Recorder:** the four counts, why scikit-learn uses that order, and which cell is the approved defaulters.

<details><summary>Hint 1</summary>

The first part of each mask is the prediction (`y_pred`), the second is the truth (`y_test`). A false positive is predicted 1 but really 0.

</details>

<details><summary>Hint 2</summary>

`FP = ((y_pred == 1) & (y_test == 0)).sum()`. FN and TN follow the same pattern.

</details>

In [ ]:
TP = ((y_pred == 1) & (y_test == 1)).sum()   # predicted default, really defaulted
FP = np.nan    # ✏️ predicted default, really repaid
FN = np.nan    # ✏️ predicted repay, really defaulted
TN = np.nan    # ✏️ predicted repay, really repaid

print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}   total={TP + FP + FN + TN}")
if np.isnan([FP, FN, TN]).any():
    print('Fill in FP, FN and TN to see scikit-learn\'s counts.')
else:
    print('scikit-learn (tn, fp, fn, tp):', confusion_matrix(y_test, y_pred).ravel())

## Q4 · Precision, recall and F1 by hand
> **Quiz Q4:** Replace the two `np.nan`s with the formulas for precision and recall, and run the cell. Say what precision and recall mean for this bank, using the counts ("of the ... applicants the model flagged, ..."). The model flags exactly 90 applicants as defaults, the same as the number of applicants who really defaulted. What does that have to do with precision and recall coming out equal?

**✏️ Your change:** Write precision and recall using `tp`, `fp` and `fn` (they are already in the cell).

**Tell your Recorder:** the three numbers, what precision and recall mean in counts, and why flagging 90 makes them equal.

<details><summary>Hint 1</summary>

Your answer to Q2(a) says what goes in each denominator.

</details>

<details><summary>Hint 2</summary>

`precision = tp / (tp + fp)` and `recall = tp / (tp + fn)`

</details>

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

precision = np.nan   # ✏️
recall = np.nan      # ✏️
f1 = 2 * precision * recall / (precision + recall)

print(f"by hand:      precision {precision:.3f}   recall {recall:.3f}   F1 {f1:.3f}")
print(f"flagged as defaults: {tp + fp}   really defaulted: {tp + fn}")
if not np.isnan(f1):
    print(f"scikit-learn: precision {precision_score(y_test, y_pred):.3f}   "
          f"recall {recall_score(y_test, y_pred):.3f}   F1 {f1_score(y_test, y_pred):.3f}")

## Q5 · Accuracy picks a model
> **Quiz Q5:** 18_1_1 trained the model with `scale_pos_weight` ≈ 2.33 so that mistakes on defaulters count more during training: the weight-2.33 model. The Q5 cell compares two models, but right now both use that weight. Change the second weight to 1 (no extra weight on defaulters) and run it. Which model has the higher accuracy, and by how many applicants out of 300? Which catches more defaulters? {COST_NOTE} Which model costs the bank less? Your manager wants to deploy the model with the higher accuracy. What do you tell them?

**✏️ Your change:** Change the second weight in the line marked ✏️ from `SPW` to `1`, then run the cell.

**Tell your Recorder:** the two accuracies (and the gap in applicants), the two recalls, the two costs, then your reply to the manager.

<details><summary>Hint 1</summary>

`SPW` is the 2.33 weight from Setup. A weight of 1 means a defaulter's mistake counts the same as anyone else's.

</details>

<details><summary>Hint 2</summary>

`weights = [SPW, 1]`

</details>

In [ ]:
weights = [SPW, SPW]     # ✏️ change the second weight to 1: no extra weight on defaulters

rows = []
for w in weights:
    name = f'weight {w:.2f}'
    pred = make_model(w).fit(X_train, y_train).predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({'model': name, 'correct (of 300)': (pred == y_test).sum(), 'accuracy': accuracy_score(y_test, pred),
                 'precision': precision_score(y_test, pred), 'recall': recall_score(y_test, pred),
                 'F1': f1_score(y_test, pred), 'FN': fn, 'FP': fp, 'cost (5*FN + FP)': 5 * fn + fp})
pd.DataFrame(rows).set_index('model').round(3)

## Q6 · Read the classification report
> **Quiz Q6:** The Q6 cell prints the classification report for the weight-2.33 model. Change it to the weight-1 model from Q5 and run it. In plain words, what do the two recall values mean (the Non-Default row and the Default row)? The macro average F1 and the weighted average F1 differ: which is higher, and why? Does either average tell a bank that cares most about catching defaults what it needs to know, or would you point to a specific row?

**✏️ Your change:** Change `y_pred` to `y_pred_w1` in the `classification_report(...)` line.

**Tell your Recorder:** what each recall means in counts, which average is higher and why, and what you would show the bank.

<details><summary>Hint 1</summary>

`support` is how many test applicants are in each row: 210 repaid, 90 defaulted.

</details>

<details><summary>Hint 2</summary>

`classification_report(y_test, y_pred_w1, ...)`

</details>

In [ ]:
y_pred_w1 = make_model(1).fit(X_train, y_train).predict(X_test)

print(classification_report(y_test, y_pred,    # ✏️ use the weight-1 model's predictions
                            target_names=['Non-Default (0)', 'Default (1)']))

# Extra credit (Q7-Q8): Beyond the Default Settings
Only if you have finished Q1-Q6.

## ⭐ Q7 · Move the threshold
> **Quiz Q7:** Every number so far used the 0.5 threshold. In the Q7 cell, lower `THRESHOLD` to 0.3 for the weight-2.33 model and run it. What happens to recall, precision, accuracy and cost? Accuracy is now below the 70% always-repay baseline. Is this model worse than the one at 0.5?

**✏️ Your change:** Change `THRESHOLD = 0.5` to `THRESHOLD = 0.3`, then run the cell.

**Tell your Recorder:** recall, precision, accuracy and cost at 0.3, then whether the model is worse.

<details><summary>Hint 1</summary>

A lower threshold flags anyone whose predicted probability of default is at least 0.3, so more applicants get flagged.

</details>

<details><summary>Hint 2</summary>

`THRESHOLD = 0.3`

</details>

In [ ]:
THRESHOLD = 0.5     # ✏️ try 0.3

pred = (y_proba >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"threshold {THRESHOLD}:  recall {recall_score(y_test, pred):.3f}   "
      f"precision {precision_score(y_test, pred):.3f}   accuracy {accuracy_score(y_test, pred):.3f}")
print(f"FN {fn}   FP {fp}   cost (5*FN + FP) {5 * fn + fp}")

## ⭐ Q8 · The always-repay baseline
> **Quiz Q8:** 18_1_1 compared the model with a baseline that predicts "repay" for everyone. In the Q8 cell, replace `np.nan` with 300 predictions of 0 and run it. What are the baseline's accuracy, recall, precision and cost? Precision prints as 0, but it is really 0 / 0: why? What do recall and cost tell you that accuracy hides?

**✏️ Your change:** Replace `np.nan` with `np.zeros(len(y_test), dtype=int)`.

**Tell your Recorder:** the four numbers, why precision is really 0 / 0, and what accuracy hides.

<details><summary>Hint 1</summary>

`np.zeros(n, dtype=int)` makes an array of `n` zeros.

</details>

<details><summary>Hint 2</summary>

`baseline = np.zeros(len(y_test), dtype=int)`

</details>

In [ ]:
baseline = np.nan      # ✏️ 300 predictions of 0 ("repay")

tn, fp, fn, tp = confusion_matrix(y_test, baseline).ravel()
print(f"accuracy {accuracy_score(y_test, baseline):.3f}   recall {recall_score(y_test, baseline):.3f}   "
      f"precision {precision_score(y_test, baseline):.3f}")
print(f"FN {fn}   FP {fp}   cost (5*FN + FP) {5 * fn + fp}")